In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, rand, when
import datetime

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("EV_Battery_Energy_Pipeline").getOrCreate()

# 2. Define the total number of EV cars and create the base DataFrame
total_ev_cars = 1000
raw_ev_df = spark.range(1, total_ev_cars + 1).withColumnRenamed("id", "Vehicle_Battery_ID")

# 3. Generate mock real-time EV Battery parameters using PySpark functions
ev_battery_dataframe = raw_ev_df \
    .withColumn("Timestamp", lit(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))) \
    .withColumn("Battery_Type", lit("Lithium-ion (Smart BMS Pack)")) \
    .withColumn("Voltage_V", (rand(42) * 100 + 300).cast("int")) \
    .withColumn("State_of_Charge_SoC_Percent", (rand(24) * 85 + 15).cast("int")) \
    .withColumn("Battery_Temperature_Celsius", (rand(85) * 45 + 15).cast("int")) \
    .withColumn("Power_Consumption_kW", (col("Voltage_V") * (rand(12) * 50) / 1000).cast("double"))

# 4. Display the top 5 records of the simulated data
ev_battery_dataframe.show(5, truncate=False)

+------------------+-------------------+----------------------------+---------+---------------------------+---------------------------+--------------------+
|Vehicle_Battery_ID|Timestamp          |Battery_Type                |Voltage_V|State_of_Charge_SoC_Percent|Battery_Temperature_Celsius|Power_Consumption_kW|
+------------------+-------------------+----------------------------+---------+---------------------------+---------------------------+--------------------+
|1                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|308      |34                         |21                         |7.452679750316973   |
|2                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|331      |90                         |28                         |6.000339693829078   |
|3                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|306      |59                         |35                         |11.327932691343053  |
|4                 |2026-07-02 14:14:57|Lithium-ion (Smart

In [0]:
# 1. Define safety thresholds for the EV battery pack
MAX_SAFE_TEMPERATURE_CELSIUS = 45
MIN_SAFE_SOC_PERCENT = 20

# 2. Filter critical records where battery is overheating OR battery level is too low
critical_battery_alerts_df = ev_battery_dataframe.filter(
    (col("Battery_Temperature_Celsius") > MAX_SAFE_TEMPERATURE_CELSIUS) | 
    (col("State_of_Charge_SoC_Percent") < MIN_SAFE_SOC_PERCENT)
)

# 3. Add a dynamic status column to categorize the specific alert type
final_alerts_df = critical_battery_alerts_df.withColumn(
    "Alert_Status",
    when(col("Battery_Temperature_Celsius") > MAX_SAFE_TEMPERATURE_CELSIUS, "CRITICAL_OVERHEAT")
    .when(col("State_of_Charge_SoC_Percent") < MIN_SAFE_SOC_PERCENT, "LOW_BATTERY_WARNING")
    .otherwise("NORMAL")
)

# 4. Display the top 10 rows of critical vehicle alerts
final_alerts_df.show(10, truncate=False)

# 5. Print the total count of vehicles requiring immediate attention
print(f"Total active critical anomalies detected: {final_alerts_df.count()}")


+------------------+-------------------+----------------------------+---------+---------------------------+---------------------------+--------------------+-------------------+
|Vehicle_Battery_ID|Timestamp          |Battery_Type                |Voltage_V|State_of_Charge_SoC_Percent|Battery_Temperature_Celsius|Power_Consumption_kW|Alert_Status       |
+------------------+-------------------+----------------------------+---------+---------------------------+---------------------------+--------------------+-------------------+
|5                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|300      |19                         |43                         |12.995281680568716  |LOW_BATTERY_WARNING|
|6                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|330      |82                         |46                         |11.826252304451847  |CRITICAL_OVERHEAT  |
|7                 |2026-07-02 14:14:57|Lithium-ion (Smart BMS Pack)|308      |38                         |52      

In [0]:
# 1. Define the storage location/table name for the Gold layer
gold_table_name = "ev_critical_battery_alerts"

# 2. Save the final alerts dataframe as a Delta Table
final_alerts_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table_name)

print(f"Success! Gold layer table '{gold_table_name}' has been safely saved to Delta Lake.")



Success! Gold layer table 'ev_critical_battery_alerts' has been safely saved to Delta Lake.


In [0]:
 # 1. Create a temporary SQL view from the Gold Delta Table
final_alerts_df.createOrReplaceTempView("v_ev_battery_analytics")

# Query A: Find the total number of vehicles under each alert category
print("=== ANALYSIS 1: Count of Vehicles per Alert Status ===")
spark.sql("""
    SELECT Alert_Status, COUNT(Vehicle_Battery_ID) as Total_Vehicles
    FROM v_ev_battery_analytics
    GROUP BY Alert_Status
""").show()

# Query B: Identify the top 5 vehicles with the highest risk (Highest Temperature)
print("=== ANALYSIS 2: Top 5 Highest Risk Overheating Vehicles ===")
spark.sql("""
    SELECT Vehicle_Battery_ID, Battery_Temperature_Celsius, State_of_Charge_SoC_Percent
    FROM v_ev_battery_analytics
    WHERE Alert_Status = 'CRITICAL_OVERHEAT'
    ORDER BY Battery_Temperature_Celsius DESC
    LIMIT 5
""").show()

# Query C: Calculate the average power consumption for cars with low battery vs overheating
print("=== ANALYSIS 3: Average Power Consumption by Alert Type ===")
spark.sql("""
    SELECT Alert_Status, ROUND(AVG(Power_Consumption_kW), 2) as Avg_Power_kW
    FROM v_ev_battery_analytics
    GROUP BY Alert_Status
""").show()


=== ANALYSIS 1: Count of Vehicles per Alert Status ===
+-------------------+--------------+
|       Alert_Status|Total_Vehicles|
+-------------------+--------------+
|LOW_BATTERY_WARNING|            43|
|  CRITICAL_OVERHEAT|           306|
+-------------------+--------------+

=== ANALYSIS 2: Top 5 Highest Risk Overheating Vehicles ===
+------------------+---------------------------+---------------------------+
|Vehicle_Battery_ID|Battery_Temperature_Celsius|State_of_Charge_SoC_Percent|
+------------------+---------------------------+---------------------------+
|                62|                         59|                         45|
|                22|                         59|                         80|
|                91|                         59|                         25|
|               209|                         59|                         83|
|               172|                         59|                         29|
+------------------+--------------------------

In [0]:

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import when, col

# 1. Prepare Target Label (Convert Alert_Status text into numerical values for AI)
# 0 = NORMAL/LOW_BATTERY, 1 = CRITICAL_OVERHEAT (Target to predict)
ml_input_df = final_alerts_df.withColumn(
    "Label", 
    when(col("Alert_Status") == "CRITICAL_OVERHEAT", 1).otherwise(0)
)

# 2. Feature Engineering: Group all input features that the AI should learn from
feature_columns = ["Voltage_V", "State_of_Charge_SoC_Percent", "Battery_Temperature_Celsius", "Power_Consumption_kW"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="Features")
ml_dataset = assembler.transform(ml_input_df)

# 3. Split dataset into Training (80%) and Testing (20%) sets
train_data, test_data = ml_dataset.randomSplit([0.8, 0.2], seed=42)

# 4. Initialize and Train the Random Forest Classifier Algorithm
rf_classifier = RandomForestClassifier(featuresCol="Features", labelCol="Label", numTrees=20)
rf_model = rf_classifier.fit(train_data)

# 5. Make AI Predictions on the Test Dataset
predictions_df = rf_model.transform(test_data)

# 6. Evaluate AI Model Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="Label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions_df)

print("==================================================")
print(f"AI Machine Learning Model Training Successful!")
print(f"Random Forest Model Accuracy: {round(accuracy * 100, 2)}%")
print("==================================================")

# 7. Display AI Predictions (Actual Label vs AI Predicted Label)
predictions_df.select("Vehicle_Battery_ID", "Battery_Temperature_Celsius", "Label", "prediction").show(10)

AI Machine Learning Model Training Successful!
Random Forest Model Accuracy: 98.63%
+------------------+---------------------------+-----+----------+
|Vehicle_Battery_ID|Battery_Temperature_Celsius|Label|prediction|
+------------------+---------------------------+-----+----------+
|                 7|                         52|    1|       1.0|
|                18|                         56|    1|       1.0|
|                26|                         58|    1|       1.0|
|                40|                         48|    1|       1.0|
|                56|                         51|    1|       1.0|
|                69|                         55|    1|       1.0|
|                92|                         57|    1|       1.0|
|               115|                         55|    1|       1.0|
|               129|                         57|    1|       1.0|
|               143|                         57|    1|       1.0|
+------------------+---------------------------+-----+----